# 01 · Dataset audit and leak-free splits

What this notebook does (about 3-5 minutes, uses the GPU for the pixel comparison):
1. MD5 of all 58,549 files → exact byte copies.
2. For each byte-unique image: a rotation/flip-invariant perceptual hash and a 48x48 grayscale copy.
3. **Source groups**: two images are the same source patch if their invariant hashes match **or** their pixels match
   (mean absolute difference < 6/255) under any rotation/flip. Only images of the same class are compared.
4. Shows how much a random file split (as in the old notebook) leaks into the test set.
5. Writes the 5-fold, group-aware split that **every** model uses: `outputs/splits.csv` (+ SHA-256).

Outputs (in `01_dataset_audit/outputs/`): `files.csv`, `unique.csv`, `splits.csv`, `splits.csv.sha256`,
`fold_summary.csv`, `audit.json`, `audit_report.md`, figures.

In [ ]:
import sys
from pathlib import Path

ROOT = Path.cwd()
while not (ROOT / "PLAN.md").exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))

import json
from concurrent.futures import ThreadPoolExecutor

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from common import audit, config, env

config.ensure_dirs()
OUT = config.AUDIT_DIR
device = env.require_cuda()

## 1. Files and exact (byte-identical) copies

In [ ]:
files = audit.add_md5(audit.list_files())
print(f"{len(files):,} image files")
files.groupby("cls").agg(files=("path", "count"), byte_unique=("md5", "nunique"))

## 2. Byte-unique images and their features

In [ ]:
files = files.sort_values("path").reset_index(drop=True)
uniq = files.drop_duplicates("md5").reset_index(drop=True).copy()
uniq["n_copies"] = uniq["md5"].map(files["md5"].value_counts())
uniq["label"] = uniq["cls"].map(config.CLASS_TO_IDX)
assert uniq["n_copies"].sum() == len(files)

with ThreadPoolExecutor(8) as ex:
    feats = list(ex.map(audit.image_features, (config.DATA_DIR / p for p in uniq["path"])))
uniq["w"] = [f["w"] for f in feats]
uniq["h"] = [f["h"] for f in feats]
uniq["phash_inv"] = [f["phash_inv"] for f in feats]
pix = np.stack([f["pix"] for f in feats])
print(f"{len(uniq):,} byte-unique images; sizes:", uniq.groupby(["w", "h"]).size().sort_values(ascending=False).head(5).to_dict())

## 3. Source groups (near duplicates: rotations, flips, colour/brightness copies)

In [ ]:
group_id, merge_stats = audit.assign_groups(uniq, pix)
uniq["group_id"] = group_id
cross = audit.cross_class_matches(uniq, files)
print("merges:", merge_stats)
print("duplicates carrying two different labels:", cross)

table = pd.DataFrame({
    "files": files.groupby("cls").size(),
    "byte_unique": uniq.groupby("cls").size(),
    "phash_groups": uniq.groupby("cls")["phash_inv"].nunique(),
    "source_groups": uniq.groupby("cls")["group_id"].nunique(),
}).reindex(config.CLASSES)
table.loc["Total"] = table.sum()
table["files_per_source"] = (table["files"] / table["source_groups"]).round(1)
table

In [ ]:
# A few groups: every image shown is a different file of the SAME source patch
rng = np.random.default_rng(0)
fig, axes = plt.subplots(5, 8, figsize=(12, 7.8))
for r, c in enumerate(config.CLASSES):
    gs = uniq[uniq.cls == c].groupby("group_id")
    big = [g for g, d in gs if len(d) >= 4]
    g = rng.choice(big) if big else uniq[uniq.cls == c]["group_id"].iloc[0]
    members = uniq[uniq.group_id == g]["path"].tolist()[:8]
    for k in range(8):
        ax = axes[r, k]
        ax.axis("off")
        if k < len(members):
            ax.imshow(plt.imread(config.DATA_DIR / members[k]))
    axes[r, 0].set_title(f"{config.PRETTY[c]} (group {g})", fontsize=9, loc="left")
plt.suptitle("Different files, same source patch", y=1.0)
plt.tight_layout()
plt.savefig(OUT / "duplicate_examples.png", dpi=130, bbox_inches="tight")
plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(7, 3.2))
files_per_group = files.merge(uniq[["md5", "group_id"]], on="md5").groupby("group_id").size()
ax.hist(files_per_group, bins=60, color="#4c78a8")
ax.set_xlabel("files per source patch")
ax.set_ylabel("source patches")
ax.set_yscale("log")
ax.set_title("How many copies each distinct leaf patch has")
plt.tight_layout()
plt.savefig(OUT / "copies_per_source.png", dpi=130)
plt.show()

## 4. How much does a random split leak?
The old notebook split the 58,549 files at random (75/15/10). Here we repeat that split 5 times and count how many
test files have a copy of the same source patch in the training files.

In [ ]:
files = files.merge(uniq[["md5", "group_id"]], on="md5", how="left", validate="many_to_one")
leaks = []
for s in range(5):
    r = np.random.default_rng(s).permutation(len(files))
    n_test = int(0.10 * len(files))
    test, train = files.iloc[r[:n_test]], files.iloc[r[n_test + int(0.15 * len(files)):]]
    leaks.append({"seed": s,
                  "test_files_with_exact_copy_in_train": float(test["md5"].isin(set(train["md5"])).mean()),
                  "test_files_with_same_source_in_train": float(test["group_id"].isin(set(train["group_id"])).mean())})
leak = pd.DataFrame(leaks)
leak.describe().loc[["mean", "min", "max"]]

## 5. Group-aware 5-fold split (used by every model)

In [ ]:
splits = audit.make_splits(uniq, n_folds=config.N_FOLDS)
fold_summary = audit.check_splits(uniq, splits)   # raises if anything leaks
print("No source group appears in more than one of train/val/test, in any fold. Every image is tested exactly once.")
fold_summary

## 6. Save

In [ ]:
files[["path", "cls", "md5", "group_id"]].to_csv(OUT / "files.csv", index=False)
uniq[["path", "cls", "label", "md5", "n_copies", "w", "h", "phash_inv", "group_id"]].to_csv(OUT / "unique.csv", index=False)
splits.to_csv(OUT / "splits.csv", index=False)
digest = audit.sha256(OUT / "splits.csv")
(OUT / "splits.csv.sha256").write_text(f"{digest}  splits.csv\n")
fold_summary.to_csv(OUT / "fold_summary.csv", index=False)

summary = {
    "files": int(len(files)), "byte_unique": int(len(uniq)), "source_groups": int(uniq.group_id.nunique()),
    "per_class": table.drop(index="Total").to_dict(orient="index"),
    "merge_stats": merge_stats, "cross_class": cross,
    "random_split_leak": leak.mean().drop("seed").to_dict(),
    "pixel_threshold": audit.PIXEL_THRESHOLD, "n_folds": config.N_FOLDS, "splits_sha256": digest,
    "meta": env.run_meta(),
}
env.write_json(OUT / "audit.json", summary)

lines = ["# Dataset audit (JMuBEN, compare/dataset-coffee-leaves-images/Original)", "",
         f"- Files: **{len(files):,}**; byte-unique: **{len(uniq):,}**; distinct source patches: "
         f"**{uniq.group_id.nunique():,}**.",
         f"- Random 75/15/10 file split (as in the old notebook): "
         f"**{100 * leak.test_files_with_exact_copy_in_train.mean():.1f}%** of test files have a byte-identical copy in "
         f"train and **{100 * leak.test_files_with_same_source_in_train.mean():.1f}%** have the same source patch in train.",
         f"- Duplicates with conflicting labels: {cross}.",
         f"- Splits: StratifiedGroupKFold, k={config.N_FOLDS}, seed {config.SEED}; SHA-256 `{digest}`.", "",
         table.to_markdown(), "", "Per fold:", "", fold_summary.to_markdown(index=False), ""]
(OUT / "audit_report.md").write_text("\n".join(lines), encoding="utf-8")
print("\n".join(lines))
print("\nSaved to", OUT)